<a href="https://colab.research.google.com/github/DURGAPRASAD-67/-ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install -q google-genai pydantic

In [3]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [4]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [5]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [10]:
with open(r'/content/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []

for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)

        print(
            f'\nRésumé {i+1}: {parsed.name} — '
            f'{len(parsed.skills)} skills, '
            f'{parsed.experience_years} years exp'
        )

    except Exception as e:
        print(
            f'\nRésumé {i+1}: FAILED — '
            f'{type(e).__name__}: {str(e)[:200]}'
        )

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 5 sample résumés

Résumé 1: Ravi Kumar — 6 skills, 0.25 years exp

Résumé 2: Sneha Reddy — 6 skills, 0.115 years exp

Résumé 3: Arun Pillai — 8 skills, 0.5 years exp

=== Full first result ===
{
  "name": "Ravi Kumar",
  "email": "ravi.kumar@example.com",
  "phone": "+91-9876543210",
  "education": [
    {
      "degree": "B.Tech Computer Science",
      "institution": "Aditya University",
      "year": 2026
    },
    {
      "degree": "Intermediate",
      "institution": "Narayana Junior College",
      "year": 2022
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "Git",
    "Linux",
    "REST APIs"
  ],
  "projects": [
    "Built a Flask REST API for college placement portal (3-week internship at startup)",
    "Solved 250+ DSA problems on LeetCode (Top 5% in college)",
    "Final-year project: ML model for crop yield prediction (Random Forest, 87% accuracy)"
  ],
  "experience_years": 0.25
}


In [11]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":null,"education":[{"degree":"Master of Science in Computer Science","institution":"University of Example","year":2020},{"degree":"Bachelor of Engineering in Software Engineering","institution":"Tech Institute","year":2018}],"skills":["Python","Java","C++","SQL","AWS","Docker","Kubernetes","Machine Learning"],"projects":["Project Alpha","Project Beta","Project Gamma"],"experience_years":5.5}


In [12]:
with open(r'/content/Text-Resume-Video-Product-Assistant.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []

for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)

        print(
            f'\nRésumé {i+1}: {parsed.name} — '
            f'{len(parsed.skills)} skills, '
            f'{parsed.experience_years} years exp'
        )

    except Exception as e:
        print(
            f'\nRésumé {i+1}: FAILED — '
            f'{type(e).__name__}: {str(e)[:200]}'
        )

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 1 sample résumés

Résumé 1: Kristen Connelly — 4 skills, 3.0 years exp

=== Full first result ===
{
  "name": "Kristen Connelly",
  "email": "placeholder@example.com",
  "phone": "3868683442",
  "education": [
    {
      "degree": "BA in Film and Television",
      "institution": "Boston University",
      "year": 2025
    },
    {
      "degree": "Advanced Course in Digital Video Editing",
      "institution": "ADMEC Multimedia Institute",
      "year": 2018
    },
    {
      "degree": "Advanced Course in Digital Video Editing",
      "institution": "ADMEC Multimedia Institute",
      "year": 2015
    }
  ],
  "skills": [
    "Call Sheets & Sides",
    "Adobe Premiere Pro",
    "Camera Boom, Light Boom, Mic Boom",
    "DaVinci Resolve"
  ],
  "projects": [],
  "experience_years": 3.0
}


In [13]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890","education":[{"degree":"Master of Science in Computer Science","institution":"University of Example","year":2022},{"degree":"Bachelor of Science in Software Engineering","institution":"Another University","year":2020}],"skills":["Python","Java","JavaScript","React","SQL","Cloud Computing","Machine Learning"],"projects":["E-commerce Platform Development","AI-powered Chatbot","Mobile App for Task Management"],"experience_years":3.5}
